# Colab 9a — Flatten Pooling + Perturbation-based Eval

**Iteration on colab9.** Colab9 failed: Spearman ρ=0.26, Recall@5=0.049 (essentially random retrieval). Two changes here:

## What changed and why

**1. Architecture: `AvgPool → Flatten`**
- Old (colab8/colab9): `x.mean(dim=2)` collapses 8 positions into 1 vector.
- New: `x.flatten(1)` keeps all 8 positions, then `Linear(64*8 → 128)` learns position-aware combinations.
- **Why:** AvgPool worked for binary length-5/8 because only 32/256 unique sequences exist — the model can essentially memorize. For length-8 over 20 letters (26B possible sequences) the model needs position info to discriminate between random sequences. Averaging throws this away.

**2. Eval: random corpus → perturbation-planted corpus**
- Old: 1000 random AA sequences. Pairwise distances all 4..8 — no close neighbors exist by chance, so Recall@k tests an impossible task.
- New: 200 seed sequences + their k-edit perturbations (k=1,2,3) planted in the corpus. For each query, the *planted relatives* are the ground-truth neighbors. This is the actual retrieval task we care about.

Architecture parameters bump from ~18K to ~83K — still small, still trainable on Colab in a few minutes.

## 1. Setup & Imports

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Alphabet, Levenshtein, and Helpers

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
VOCAB_SIZE = len(AA_ALPHABET)
SEQ_LEN = 8

CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

def encode(seq):
    return torch.tensor([CHAR_TO_IDX[c] for c in seq], dtype=torch.long)

def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

rng = np.random.default_rng(SEED)

def random_seq(length=SEQ_LEN):
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def perturb(seq, k):
    s = list(seq)
    for _ in range(k):
        op = rng.choice(['sub', 'ins', 'del'])
        if len(s) == 0:
            op = 'ins'
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in AA_ALPHABET if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(list(AA_ALPHABET)))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    if len(s) < SEQ_LEN:
        s = s + [rng.choice(list(AA_ALPHABET)) for _ in range(SEQ_LEN - len(s))]
    elif len(s) > SEQ_LEN:
        s = s[:SEQ_LEN]
    return ''.join(s)

print(f'Alphabet: {AA_ALPHABET}')
print(f'Vocab size: {VOCAB_SIZE}, Sequence length: {SEQ_LEN}')

## 3. Training Pair Generation (unchanged from colab9)

Hybrid sampling: 50% random pairs (high-distance signal) + 50% seed/k-edit perturbation pairs (low-distance signal).

In [ ]:
CORPUS_SIZE = 5000
corpus = list(set(random_seq() for _ in range(CORPUS_SIZE)))
print(f'Training corpus size: {len(corpus)}')

N_PAIRS = 40000
n_random = N_PAIRS // 2
n_perturb = N_PAIRS - n_random

pairs = []
for _ in range(n_random):
    i, j = rng.integers(0, len(corpus), size=2)
    a, b = corpus[i], corpus[j]
    pairs.append((a, b, levenshtein(a, b) / SEQ_LEN))
for _ in range(n_perturb):
    seed = corpus[rng.integers(0, len(corpus))]
    k = int(rng.integers(0, 5))
    p = perturb(seed, k)
    pairs.append((seed, p, levenshtein(seed, p) / SEQ_LEN))

print(f'Total training pairs: {len(pairs)}')
norm_dists = [p[2] for p in pairs]
plt.figure(figsize=(8, 3))
plt.hist(norm_dists, bins=np.arange(0, 1.05, 1/SEQ_LEN), edgecolor='k', alpha=0.75)
plt.xlabel('Normalized Levenshtein distance')
plt.ylabel('Count')
plt.title('Training-pair distance distribution')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Dataset & DataLoader

In [ ]:
class SequencePairDataset(Dataset):
    def __init__(self, pairs, oversample_threshold=0.4, oversample_factor=3):
        self.data = []
        for seq_a, seq_b, norm_dist in pairs:
            entry = (encode(seq_a), encode(seq_b), torch.tensor(norm_dist, dtype=torch.float32))
            self.data.append(entry)
            if norm_dist < oversample_threshold:
                for _ in range(oversample_factor - 1):
                    self.data.append(entry)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

dataset = SequencePairDataset(pairs)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)
print(f'Dataset size (with oversampling): {len(dataset)}')

## 5. Architecture — Flatten instead of AvgPool

**The key change.** After two Conv1D layers we have shape `(batch, 64, 8)`. Old code: `mean(dim=2) → (batch, 64)`. New code: `flatten(1) → (batch, 512)`. The Linear then learns a *position-aware* combination instead of treating positions as exchangeable.

Parameter count goes from ~18K to ~83K — still very small.

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=32, hidden1=32, hidden2=64,
                 seq_len=SEQ_LEN, out_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        # CHANGED: flatten retains position info; linear input grows to hidden2 * seq_len
        self.fc = nn.Linear(hidden2 * seq_len, out_dim)

    def forward(self, x):
        x = self.embedding(x)              # (B, L, embed_dim)
        x = x.permute(0, 2, 1)              # (B, embed_dim, L)
        x = F.relu(self.conv1(x))           # (B, hidden1, L)
        x = F.relu(self.conv2(x))           # (B, hidden2, L)
        x = x.flatten(1)                    # (B, hidden2 * L)  — was: x.mean(dim=2)
        x = self.fc(x)                      # (B, out_dim)
        x = F.normalize(x, p=2, dim=1)
        return x


class SiameseNetwork(nn.Module):
    def __init__(self, **kw):
        super().__init__()
        self.encoder = SiameseEncoder(**kw)
    def forward(self, a, b):
        return torch.norm(self.encoder(a) - self.encoder(b), p=2, dim=1)
    def encode(self, x):
        return self.encoder(x)

model = SiameseNetwork().to(device)
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Loss & Training

In [ ]:
def weighted_mse_loss(pred_dist, true_dist, alpha=3.0):
    weight = torch.exp(-alpha * true_dist)
    return torch.mean(weight * (pred_dist - true_dist) ** 2)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 100
losses = []

model.train()
for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    nb = 0
    for seq_a, seq_b, true_dist in dataloader:
        seq_a = seq_a.to(device); seq_b = seq_b.to(device); true_dist = true_dist.to(device)
        pred = model(seq_a, seq_b)
        loss = weighted_mse_loss(pred, true_dist)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item(); nb += 1
    avg = epoch_loss / nb
    losses.append(avg)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{num_epochs} — Loss: {avg:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Weighted MSE Loss')
plt.title('Training Loss — flatten architecture')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f'Final loss: {losses[-1]:.6f}')

## 7. Perturbation-planted Eval Corpus

Build a corpus where ground-truth retrieval is meaningful:
- 200 **seeds** (queries) — fresh random sequences not in training
- For each seed, generate **3 perturbations** at k=1, k=2, k=3
- Pad with **distractors** (1000 random sequences) so retrieval is non-trivial

For each seed query, the planted (k=1, k=2, k=3) versions are the ground-truth top-3 neighbors.

In [ ]:
N_SEEDS = 200
K_VALUES = [1, 2, 3]
N_DISTRACTORS = 1000

seeds = [random_seq() for _ in range(N_SEEDS)]
# For each seed: planted relatives at k=1,2,3
planted = []  # list of (seed_idx, k, sequence)
for si, seed in enumerate(seeds):
    for k in K_VALUES:
        planted.append((si, k, perturb(seed, k)))

distractors = [random_seq() for _ in range(N_DISTRACTORS)]

# Eval corpus: seeds + planted relatives + distractors
eval_corpus = seeds + [p[2] for p in planted] + distractors
# Track which corpus indices are the planted relatives of each seed
# seeds occupy [0, N_SEEDS); planted occupy [N_SEEDS, N_SEEDS + len(planted))
seed_to_planted_indices = {si: [] for si in range(N_SEEDS)}
for offset, (si, k, _) in enumerate(planted):
    seed_to_planted_indices[si].append(N_SEEDS + offset)

print(f'Eval corpus: {len(eval_corpus)} (seeds={N_SEEDS}, planted={len(planted)}, distractors={N_DISTRACTORS})')

# Embed everything
model.eval()
tensors = torch.stack([encode(s) for s in eval_corpus]).to(device)
with torch.no_grad():
    eval_embeddings = model.encode(tensors).cpu().numpy()
emb_tensor = torch.tensor(eval_embeddings, dtype=torch.float32)
emb_dists = torch.cdist(emb_tensor, emb_tensor, p=2).numpy()
print(f'Embedding matrix: {eval_embeddings.shape}')

## 8. Retrieval Recall — Can the model find the planted relatives?

For each seed query, take the top-k predicted neighbors (excluding the seed itself). Count how many of the 3 planted relatives appear in the top-k.

- **Recall@3:** of top-3 neighbors, how many are the 3 planted relatives? Perfect = 3/3 = 1.0.
- **Recall@10:** more lenient — at least 3/10 should be planted relatives.

We also break this down by perturbation distance — *can* the model find the k=1 relative? The k=3?

In [ ]:
def topk_neighbors(query_idx, k):
    d = emb_dists[query_idx].copy()
    d[query_idx] = np.inf
    return np.argsort(d)[:k]

for k_retrieval in [3, 5, 10]:
    recalls = []
    for si in range(N_SEEDS):
        topk = set(topk_neighbors(si, k_retrieval))
        planted_set = set(seed_to_planted_indices[si])
        recalls.append(len(topk & planted_set) / len(planted_set))
    print(f'Mean planted-recall@{k_retrieval}: {np.mean(recalls):.4f}')

# Per-perturbation-distance breakdown: rank of each planted relative in the predicted nearest list
rank_by_k = {1: [], 2: [], 3: []}
for si in range(N_SEEDS):
    d = emb_dists[si].copy()
    d[si] = np.inf
    sorted_idx = np.argsort(d)
    rank_lookup = {idx: r for r, idx in enumerate(sorted_idx)}
    for offset, planted_idx in enumerate(seed_to_planted_indices[si]):
        k_pert = K_VALUES[offset]
        rank_by_k[k_pert].append(rank_lookup[planted_idx])

print('\nMedian rank of planted relative (lower = better, 0 = top of list):')
for k_pert in K_VALUES:
    ranks = rank_by_k[k_pert]
    print(f'  k={k_pert} edits: median rank = {int(np.median(ranks))}, mean = {np.mean(ranks):.1f}, '
          f'in top-10 = {np.mean(np.array(ranks) < 10):.2%}')

## 9. Distance Correlation (full eval corpus)

Same plot as colab9 §9 — true Levenshtein vs. embedding L2 distance, sampled over the eval corpus. With the planted-relatives corpus we now actually have small-distance pairs in the eval set.

In [ ]:
n_eval = len(eval_corpus)
MAX_EVAL_PAIRS = 50000
all_ij = [(i, j) for i in range(n_eval) for j in range(i + 1, n_eval)]
if len(all_ij) > MAX_EVAL_PAIRS:
    sampled_idx = rng.choice(len(all_ij), size=MAX_EVAL_PAIRS, replace=False)
    all_ij = [all_ij[k] for k in sampled_idx]

true_d, pred_d = [], []
for i, j in all_ij:
    true_d.append(levenshtein(eval_corpus[i], eval_corpus[j]) / SEQ_LEN)
    pred_d.append(emb_dists[i, j])
true_d = np.array(true_d); pred_d = np.array(pred_d)

pr, _ = pearsonr(true_d, pred_d); sr, _ = spearmanr(true_d, pred_d)
print(f'Pearson  r = {pr:.4f}')
print(f'Spearman ρ = {sr:.4f}')

plt.figure(figsize=(7, 6))
plt.scatter(true_d, pred_d, alpha=0.05, s=5)
coeffs = np.polyfit(true_d, pred_d, 1)
x_line = np.linspace(0, 1, 100)
plt.plot(x_line, np.polyval(coeffs, x_line), 'r-', linewidth=2,
         label=f'Linear fit (slope={coeffs[0]:.3f})')
plt.xlabel('True Normalized Levenshtein Distance')
plt.ylabel('Predicted Embedding L2 Distance')
plt.title(f'Distance Correlation (planted eval)\nPearson r={pr:.3f}, Spearman ρ={sr:.3f}')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 10. Embedding Visualization

PCA / t-SNE of the eval corpus, colored by Levenshtein distance to the first seed. With planted relatives in the corpus, we should see a clear gradient near the seed.

In [ ]:
ref_seq = seeds[0]
colors = [levenshtein(s, ref_seq) for s in eval_corpus]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pca_2d = PCA(n_components=2).fit_transform(eval_embeddings)
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=colors, cmap='viridis', s=15, alpha=0.7)
axes[0].set_title('PCA projection'); axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

tsne_2d = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(eval_embeddings)
sc = axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=colors, cmap='viridis', s=15, alpha=0.7)
axes[1].set_title('t-SNE projection'); axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')
fig.colorbar(sc, ax=axes, label=f"Levenshtein distance to '{ref_seq}'")
plt.suptitle('Embedding Space — eval corpus with planted relatives', fontsize=13)
plt.tight_layout(); plt.show()

## 11. Diagnostic: example query results

Print the top-5 predicted neighbors for the first 5 seed queries, alongside the planted relatives and their true Levenshtein distances. This makes failure modes obvious.

In [ ]:
for si in range(5):
    seed = seeds[si]
    print(f"\nQuery seed: '{seed}'")
    print(f"  Planted relatives:")
    for offset, planted_idx in enumerate(seed_to_planted_indices[si]):
        k_pert = K_VALUES[offset]
        rel_seq = eval_corpus[planted_idx]
        true_d_rel = levenshtein(seed, rel_seq)
        emb_d_rel = emb_dists[si, planted_idx]
        print(f"    k={k_pert}: '{rel_seq}'  true_d={true_d_rel}  emb_d={emb_d_rel:.4f}")
    print(f"  Top-5 predicted neighbors:")
    top5 = topk_neighbors(si, 5)
    for rank, idx in enumerate(top5):
        ns = eval_corpus[idx]
        marker = ' <-- PLANTED' if idx in seed_to_planted_indices[si] else ''
        print(f"    {rank+1}. '{ns}'  emb_d={emb_dists[si, idx]:.4f}  true_d={levenshtein(seed, ns)}{marker}")

## 12. Decision

**Success criteria:**
- Median rank of planted k=1 relative < 5
- planted-recall@10 > 0.6
- Spearman ρ > 0.7 (with the corrected eval that has small-distance pairs)

**If passes →** colab10 with real Swiss-Prot data, same architecture.

**If still fails:**
- Bump `embed_dim` 32 → 64 (richer per-token representation)
- Try `out_dim` 128 → 256
- Drop L2 normalization at the end (unbounded distance, may help separation)
- Inspect example queries (cell 11) — are k=1 perturbations being found at all? If not, position info still isn't reaching the embedding.